Author: Robin Sternberg  
This notebook is licensed under Creative Commons Attribution-ShareAlike 4.0

# Laden der Protokolle über die OpenData-API des Bundestags

Dieses Notebook dient zur Datenbeschaffung. Die Plenarprotokolle des Bundestags werden über die API des Dokumentations- und Informationssystems für Parlamentsmaterialien (DIP) heruntergeladen. Dokumentation gibt es [hier](https://dip.bundestag.de/%C3%BCber-dip/hilfe/api#content).  
Die Protokolle gibt es auch zum Download in ZIP-Archiven für alle abgeschlossenen Wahlperioden [hier (am unteren Seitenende)](https://www.bundestag.de/services/opendata). Protokolle der aktuellen Wahlperiode müssten aber einzeln heruntergeladen werden und stehen nur im XML-Format zur Verfügung.

# Anmerkung Formate

Die neueren Protokolle, die man im XML-Format für die aktuelle Bundestagsperiode herunterladen kann, sind schon vorverarbeitet.  
Das bedeutet hier ist in XML-Struktur angemerkt, wann eine Rede anfängt und wer die Rede hält. Dazu werden auch "kommentare" (Beifall, Zwischenrufe etc.) eigens im xml verpackt. Ein Beispiel ist [hier](https://www.bundestag.de/resource/blob/967446/cb0b3e82501a75c7c9d082b659b6a1e8/20122.xml).  
Diese Aufmachung ist enorm praktisch und erleichtert das Parsen der Protokolle ungemein. Allerdings gibt es diese aufbereitete Form erst seit dem 21.09.2023. Alle historischen Protokolle bis zum Vortag sind nicht vorverarbeitet und liegen in Plaintext-Form vor.  

Die Dateien, die über die API bereitgestellt werden, sind gänzlich nicht vorverarbeitet. Das gilt auch für die Neueren ab dem 21.09.2023 im XML-Format.
[Beispiel](https://search.dip.bundestag.de/api/v1/plenarprotokoll-text?format=xml&apikey=I9FKdCn.hbfefNWCY336dL6x62vfwNKpoN2RZ1gp21&f.datum.start=2024-03-20&f.datum.end=2024-03-23&f.zuordnung=BT)

Da das neue Format noch nicht einmal ein Jahr vorliegt, wären viele der interessanten Analysen auf den Daten witzlos, weshalb ich die nicht-vorverarbeitete Variante im Json-Format über die API gewählt habe. So konnte ich dann auch von der leichten Einbindung Json-Dateien über Python Dictionaries profitieren.

# Code

In [2]:
import requests
import pandas as pd
import json

Die Funktion `dict_to_params` erstellt aus einem Dictionary die darin spezifizierten Parameter im Format &&lt;paramname&gt;=&lt;paramvalue&gt;.  
Diese können dann an einen API-Call angehängt werden, um z.B. das Datum einzugrenzen

In [3]:
def dict_to_params(data):
    html_params = ""
    for key, value in data.items():
        if value != None:
            html_params += f"&{key}={value}"
    return html_params.strip()

In diesem Code-Block werden die Protokolle zwischen `f.datum.start` und `f.datum.end` in `../_data/protocols` geladen.  
Das geschieht über mehrere Calls an die API, da die API immer maximal zehn Protokolle auf einmal zurückliefert.  
Damit der "Fortschritt" erhalten bleibt, liefert die API im Response-Objekt einen Cursor mit, der im nächsten Call angehängt wird.  

Der Name der abgespeicherten Json-Dateien setzt sich wie folgt zusammen:  
&lt;Wahlperiode&gt;\_&lt;fortlaufende Protokollnummer&gt;\_&lt;datum(yyy-mm-dd)&gt;.json


In [7]:
url_static= "https://search.dip.bundestag.de/api/v1/plenarprotokoll-text?format=json&apikey=I9FKdCn.hbfefNWCY336dL6x62vfwNKpoN2RZ1gp21" # if there is an error, check if the api key is still valid at https://dip.bundestag.de/%C3%BCber-dip/hilfe/api
data_path = '../_data/protocols/'

params = {
        "f.datum.start": "2024-01-14", # Change here for different Dates
        "f.datum.end": "2024-06-20",
        "f.zuordnung": "BT",
        "cursor" : None
    }

doc_count = 0
while(True):
    
    api_url = url_static + dict_to_params(params)
    print(api_url)

    response = requests.get(api_url)
    print(response)

    if response.status_code == 200: # Everything fine

        doc_list = response.json()['documents']
        doc_count += len(doc_list)
        for doc in doc_list:
            path = f'{data_path}{str(doc['wahlperiode']).zfill(2)}_{doc['dokumentnummer'].split(r'/')[1].zfill(3)}_{doc['datum']}.json'
            print(path)
            with open(path, 'w', encoding='utf-8') as json_file:
                json.dump(doc,json_file, indent=4)
    else:
         print("Error in call. Aborting")
         break
    if 'cursor' not in response.json() or response.json()['cursor'] == params['cursor']:
        print(f"Retrieved all {doc_count} Documents")
        break
    else:
        params['cursor'] = response.json()['cursor']

    

https://search.dip.bundestag.de/api/v1/plenarprotokoll-text?format=json&apikey=I9FKdCn.hbfefNWCY336dL6x62vfwNKpoN2RZ1gp21&f.datum.start=2024-01-14&f.datum.end=2024-06-20&f.zuordnung=BT
<Response [200]>
../_data/protocols/20_176_2024-06-14.json
../_data/protocols/20_175_2024-06-13.json
../_data/protocols/20_174_2024-06-12.json
../_data/protocols/20_173_2024-06-07.json
../_data/protocols/20_172_2024-06-06.json
../_data/protocols/20_171_2024-06-05.json
../_data/protocols/20_170_2024-05-17.json
../_data/protocols/20_169_2024-05-16.json
../_data/protocols/20_168_2024-05-15.json
../_data/protocols/20_167_2024-04-26.json
https://search.dip.bundestag.de/api/v1/plenarprotokoll-text?format=json&apikey=I9FKdCn.hbfefNWCY336dL6x62vfwNKpoN2RZ1gp21&f.datum.start=2024-01-14&f.datum.end=2024-06-20&f.zuordnung=BT&cursor=AoJw0J72jI8DNFBsZW5hcnByb3Rva29sbC01NjQ0
<Response [200]>
../_data/protocols/20_166_2024-04-25.json
../_data/protocols/20_165_2024-04-24.json
../_data/protocols/20_164_2024-04-12.json
..